# Checklist Evidence Assistant

**Revised 2026-07-28, after all 15 papers were appraised by hand.**

This notebook does **not** score papers and no longer suggests any verdict. It reads each PDF and surfaces the passages relevant to each of the 13 checklist items, with page numbers, so the reading goes faster. Every Yes / No / Unclear / N/A in `checklist_scoring_tracker.xlsx` was assigned by a human reading the full paper.

## What changed, and why

The first version auto-suggested Yes/No for two items on the grounds that a regex match was "close to unambiguous". Appraising the papers proved otherwise, so **both suggestion columns have been withdrawn**.

- **Intention-to-Treat** was wrong in every case checked, by four distinct mechanisms. A PDF line break split `Intention-to-` from `treat` (Repici, EAGLE, Ladabaum). One PDF extracted with all spaces stripped, so `intentiontotreat` could not match (ACCESS). One paper followed the principle without ever naming it (ACCESS). One named it differently, as "intent-to-screen" (Abramoff).
- **Registered w/ CT.gov** returned No where the answer was N/A (Zhou), and its patterns would return No for two trials that *were* prospectively registered, just not on ClinicalTrials.gov (Alali on the Kuwaiti registry, RAIDERS on PACTR, both of which score Unclear).

Between them the two columns were wrong at least eight times and were never right about anything a human reading the Methods would have missed. They now produce evidence only, like the other eleven items.

## Two fixes that mattered more than the patterns

1. **Whitespace normalisation.** Every ITT false negative traced to whitespace, not vocabulary. Page text is now collapsed to single spaces before matching, and patterns separate tokens with `[-\s]*`, so `intention to treat`, `intention-to-treat` and `intentiontotreat` all match. This alone would have caught all four misses.
2. **Page one verification.** Two of the twelve original manifest entries contained the wrong paper, and two of eight later downloads failed screening. All four were visible on the first page. The notebook now emits `verification_log.csv` so filenames can be checked against intended citations before anything is scored.

## Outputs

| File | Format | Purpose |
| --- | --- | --- |
| `auto_review.xlsx` | Excel | Reading aid. Long wrapped snippets, read alongside the PDF. Contains no formulas. |
| `auto_review.csv` | CSV | Same content, for grep, diff and version control. |
| `verification_log.csv` | CSV | Page one identity check. Paste into the tracker's Screening Log. |

Nothing here ever writes to `checklist_scoring_tracker.xlsx`.

Requires `pdfplumber` and `openpyxl`.

In [ ]:
import re
import csv
from pathlib import Path
from dataclasses import dataclass

import pdfplumber
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side


def find_project_root(marker="Papers"):
    """Walk upward from the current directory until we find the folder that
    contains the Papers/ directory, that's the project root."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root (looking for a '{marker}' folder) "
        f"starting from {current}. Are you running this from inside the project?"
    )

PROJECT_ROOT = find_project_root()
PAPERS_DIR   = PROJECT_ROOT / "Papers"
TRACKER_PATH = PROJECT_ROOT / "Data" / "Tracker" / "checklist_scoring_tracker.xlsx"
OUT_DIR      = PROJECT_ROOT / "Data" / "Tracker"

XLSX_PATH    = OUT_DIR / "auto_review.xlsx"
CSV_PATH     = OUT_DIR / "auto_review.csv"
VERIFY_PATH  = OUT_DIR / "verification_log.csv"

CONTEXT_CHARS = 240        # characters of context kept around each match
MAX_SNIPPETS_PER_ITEM = 3  # cap so one item doesn't dominate the sheet
VERIFY_EXCERPT_CHARS = 600 # page-one excerpt length in the verification log

print(f"Project root : {PROJECT_ROOT}")
print(f"Papers dir   : {PAPERS_DIR}  (exists: {PAPERS_DIR.is_dir()})")
print(f"Tracker      : {TRACKER_PATH}  (exists: {TRACKER_PATH.exists()})")

## Load the paper manifest from the tracker

Reads the `Papers` sheet directly out of `tracker.xlsx`, rather than hardcoding the list a second time here, so the manifest never drifts out of sync with the tracker itself.

In [ ]:
@dataclass
class Paper:
    domain: str
    citation: str
    filename: str      # as written in the manifest, may be "Supporting/foo.pdf"
    path: Path         # fully resolved


def resolve_pdf(domain: str, filename: str) -> Path:
    """Manifest filenames are either bare (live under Papers/<Domain>/) or
    already carry a folder prefix such as 'Supporting/'. Handle both, and match
    the domain folder case-insensitively so 'ECG' finds 'ECG' or 'ecg'."""
    if "/" in filename or "\\" in filename:
        return PAPERS_DIR / filename                      # e.g. Supporting/x.pdf
    for sub in PAPERS_DIR.iterdir():                      # case-insensitive domain match
        if sub.is_dir() and sub.name.lower() == str(domain).lower():
            return sub / filename
    return PAPERS_DIR / str(domain) / filename            # fall through, will report missing


def load_paper_manifest(tracker_path: Path) -> list[Paper]:
    """Reads the Papers sheet directly, so the manifest never drifts out of sync
    with the tracker. Rows with a blank Domain are the supporting-evidence block
    at the bottom of the sheet and are deliberately skipped."""
    wb = openpyxl.load_workbook(tracker_path, data_only=True)
    sheet = wb["Papers"]
    papers = []
    for row in sheet.iter_rows(min_row=2, values_only=True):
        # columns: #, Domain, Citation, Type, Filename, Download Status, URL
        num, domain, citation, _type, filename, _status, _url = (list(row) + [None] * 7)[:7]
        if not domain or not filename:
            continue                                       # header, blank, or supporting block
        if not isinstance(num, int):
            continue                                       # section banner rows
        papers.append(Paper(domain=str(domain), citation=str(citation),
                            filename=str(filename), path=resolve_pdf(domain, filename)))
    return papers


papers = load_paper_manifest(TRACKER_PATH)
print(f"Loaded {len(papers)} scoreable papers from the tracker manifest\n")
missing = 0
for p in papers:
    ok = p.path.exists()
    missing += (not ok)
    print(f"  [{'OK ' if ok else 'MISSING'}] {p.domain:<12} {p.filename}")
if missing:
    print(f"\n!! {missing} file(s) not found. Check the Filename column against Papers/ before running.")

## Checklist item definitions

Every item is evidence only. There is no `mode` field any more, because there is no detect mode any more.

Patterns join their tokens with `W`, defined as `[-\s]*`. Since page text is whitespace-collapsed before matching, that one separator absorbs a space, a hyphen, the hyphen-plus-space left behind by a line break, or nothing at all. `intention to treat`, `intention-to-treat`, `Intention-to-\ntreat` and `intentiontotreat` therefore all match the same pattern.

These lists are meant to be tuned. If a paper phrases something in a way the patterns miss, widen them. A false positive costs you one snippet to skim. A false negative cost this project four wrong verdicts.

In [ ]:
@dataclass
class ChecklistItem:
    label: str        # must match the short label used in the tracker's Legend/Scoring tabs
    patterns: list    # list of regex patterns (case-insensitive)


# W joins tokens tolerantly. Because page text is whitespace-collapsed before
# matching, "[-\s]*" absorbs a space, a hyphen, a hyphen+space left behind by a
# line break, or nothing at all. This is what fixes the four ITT false negatives.
W = r"[-\s]*"

CHECKLIST_ITEMS = [
    ChecklistItem("External Validation", [
        rf"external{W}valid\w*", rf"external{W}cohort", rf"external{W}dataset",
        rf"external{W}benchmark", rf"held{W}out", rf"temporal{W}validation",
        rf"geograph\w*{W}validation", rf"independent{W}validation",
        rf"prospective{W}validation"]),
    ChecklistItem("Patient-Level RCT", [
        rf"cluster{W}randomi[sz]\w*", rf"patient{W}level{W}randomi[sz]\w*",
        rf"individually{W}randomi[sz]ed", rf"randomi[sz]ed{W}(?:controlled{W})?trial",
        rf"randomly{W}assigned", rf"allocation{W}conceal\w*",
        rf"historical{W}control", rf"before{W}and{W}after"]),
    ChecklistItem("Prespecified Endpoint", [
        rf"primary{W}(?:outcome|endpoint|end{W}point)",
        rf"pre{W}?specified{W}(?:primary{W})?(?:outcome|endpoint|end{W}point)",
        rf"co{W}primary{W}(?:outcome|endpoint)", rf"secondary{W}(?:outcome|endpoint)"]),
    ChecklistItem("Prespecified Design", [
        rf"study{W}protocol", rf"pre{W}?specified{W}design", rf"trial{W}design",
        rf"statistical{W}analysis{W}plan", rf"no{W}changes{W}to",
        rf"protocol{W}deviation", rf"amend\w*{W}the{W}protocol"]),
    # Was mode="detect". Now evidence only, and broadened past ClinicalTrials.gov
    # so that non-US registries surface for a human to judge (these score Unclear,
    # not No, and the old patterns could not see them at all).
    ChecklistItem("Registered w/ CT.gov", [
        rf"NCT{W}\d{{8}}", r"clinicaltrials\.gov",
        rf"ISRCTN{W}\d+", rf"PACTR{W}\d+", rf"ChiCTR[-\w]*{W}\d+",
        rf"UMIN{W}\d+", rf"ACTRN{W}\d+", rf"CRD{W}\d+",
        rf"registered{W}(?:before|prior)", rf"trial{W}registration",
        rf"registry{W}(?:number|identifier)", rf"prospectively{W}registered"]),
    ChecklistItem("Transparent Reporting", [
        rf"code{W}availab\w*", rf"data{W}availab\w*", rf"open{W}source",
        rf"proprietary", rf"black{W}box", rf"model{W}weights",
        rf"publicly{W}available", rf"upon{W}reasonable{W}request",
        rf"github", rf"architect\w*", rf"convolutional{W}neural{W}network",
        rf"feature{W}importance", rf"saliency", rf"interpretab\w*"]),
    ChecklistItem("Honest Impact Eval", [
        rf"limitations?", rf"we{W}acknowledge", rf"overestimat\w*", rf"underestimat\w*",
        rf"did{W}not{W}(?:reach|meet|show|differ)", rf"not{W}statistically{W}significant",
        rf"type{W}(?:2|II){W}error", rf"caution", rf"further{W}(?:research|study|work)",
        rf"unclear", rf"Hawthorne"]),
    ChecklistItem("No Conflict of Interest", [
        rf"conflicts?{W}of{W}interest", rf"competing{W}interests?", rf"disclosur\w*",
        rf"consultant", rf"employee", rf"shareholder", rf"equity",
        rf"speaker{W}fees", rf"licens\w*", rf"funded{W}by", rf"sponsor\w*",
        rf"role{W}of{W}the{W}(?:funder|sponsor)", rf"in{W}kind"]),
    ChecklistItem("Focus on Patient Outcomes", [
        rf"all{W}cause{W}mortality", rf"mortality", rf"morbidity", rf"survival",
        rf"detection{W}rate", rf"adenoma{W}detection{W}rate", rf"surrogate",
        rf"process{W}measure", rf"interval{W}cancer", rf"post{W}colonoscopy",
        rf"long{W}term{W}outcome", rf"downstream{W}(?:clinical{W})?impact"]),
    ChecklistItem("Adequate Sample Size", [
        rf"sample{W}size", rf"power(?:ed)?{W}(?:calculation|to{W}detect)",
        rf"statistical{W}power", rf"per{W}(?:group|arm){W}were{W}required",
        rf"alpha", rf"dropout{W}rate", rf"intracluster{W}correlation",
        rf"noninferiority{W}margin", rf"non{W}inferiority{W}margin"]),
    # Was mode="detect". Now evidence only. Patterns widened to catch the
    # variants that defeated the original: line-break splits, space-stripped
    # extraction, modified ITT, and "intent-to-screen".
    ChecklistItem("Intention-to-Treat", [
        rf"inten\w*{W}to{W}treat", rf"inten\w*{W}to{W}screen",
        rf"modified{W}inten\w*{W}to{W}treat", rf"\bmITT\b", rf"\bITT\b",
        rf"per{W}protocol", rf"as{W}randomi[sz]ed",
        rf"excluded{W}(?:from{W})?(?:the{W})?analys[ei]s", rf"post{W}randomi[sz]ation"]),
    ChecklistItem("Balanced Table 1", [
        rf"table{W}1\b", rf"baseline{W}characteristics", rf"baseline{W}demograph\w*",
        rf"similar{W}(?:between|in){W}(?:the{W})?(?:two|both){W}groups",
        rf"standardized{W}mean{W}difference", rf"did{W}not{W}differ"]),
    ChecklistItem("No Reverse Causation", [
        rf"data{W}leakage", rf"reverse{W}causation", rf"excluded{W}as{W}a{W}predictor",
        rf"training{W}(?:set|data|dataset)", rf"index{W}(?:date|time)",
        rf"locked", rf"development{W}dataset", rf"contribut\w*{W}to{W}the{W}training"]),
]

assert len(CHECKLIST_ITEMS) == 13
print(f"{len(CHECKLIST_ITEMS)} items, all evidence-only. No verdict is suggested for any item.")
for it in CHECKLIST_ITEMS:
    for pat in it.patterns:
        re.compile(pat, re.IGNORECASE)   # fail fast on a malformed pattern
print("All patterns compile.")

## PDF text extraction and evidence search

In [ ]:
def _norm(s: str) -> str:
    """Collapse every run of whitespace to a single space."""
    return re.sub(r"\s+", " ", s or "").strip()


def extract_pages(pdf_path: Path) -> list[tuple[int, list[str]]]:
    """Returns (page_number, [text_variant, ...]), 1-indexed.

    THREE extraction failure modes defeated the original notebook, and each
    needs a different remedy:

    1. Line breaks inside a phrase. "Intention-to-\\ntreat" (Repici, EAGLE,
       Ladabaum). Fixed by whitespace normalisation below.
    2. Space-stripped extraction. Some PDFs come out as "intentiontotreat"
       (ACCESS). Fixed by the [-\\s]* separator in the patterns, which can
       match zero characters.
    3. COLUMN INTERLEAVING. In a two-column journal article pdfplumber often
       reads straight across the page, splicing the left column into the right
       mid-sentence, e.g. RAIDERS extracts as "intentionto Clinic staff
       capturing and u...". Neither remedy above helps, because the words are
       genuinely far apart in the extracted string.

    So each page yields up to three text variants: the whole page as extracted,
    then the left and right halves cropped separately. A phrase severed by a
    column boundary survives in at least one of them."""
    variants_per_page = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            texts = [_norm(page.extract_text())]
            try:                       # two-column rescue
                w, h = page.width, page.height
                left = page.crop((0, 0, w / 2, h)).extract_text()
                right = page.crop((w / 2, 0, w, h)).extract_text()
                texts += [_norm(left), _norm(right)]
            except Exception:
                pass                   # single-column or odd geometry, whole page is enough
            variants_per_page.append((i, [t for t in texts if t]))
    return variants_per_page


def find_snippets(pages, patterns, max_snippets):
    """Search every text variant of every page. Returns up to max_snippets
    context windows with page numbers, deduplicated on snippet content so the
    column-crop variants do not repeat what the full-page pass already found."""
    compiled = [re.compile(p, re.IGNORECASE) for p in patterns]
    snippets, seen = [], set()
    for page_num, variants in pages:
        for text in variants:
            for regex in compiled:
                for m in regex.finditer(text):
                    start = max(0, m.start() - CONTEXT_CHARS // 2)
                    end = min(len(text), m.end() + CONTEXT_CHARS // 2)
                    snip = text[start:end].strip()
                    key = re.sub(r"\W", "", snip.lower())[:120]
                    if key in seen:
                        continue
                    seen.add(key)
                    snippets.append({"page": page_num, "text": snip})
                    if len(snippets) >= max_snippets:
                        return snippets
    return snippets


def evaluate_paper(paper: Paper) -> list[dict]:
    """Runs every checklist item against one paper. One row per item.
    No verdict is ever suggested, only evidence."""
    if not paper.path.exists():
        return [{"domain": paper.domain, "citation": paper.citation,
                 "filename": paper.filename, "item": item.label,
                 "evidence": f"FILE NOT FOUND at {paper.path}", "n_hits": 0}
                for item in CHECKLIST_ITEMS]

    pages = extract_pages(paper.path)
    rows = []
    for item in CHECKLIST_ITEMS:
        snips = find_snippets(pages, item.patterns, MAX_SNIPPETS_PER_ITEM)
        evidence = (" | ".join(f"p.{s['page']}: “...{s['text']}...”" for s in snips)
                    if snips else
                    "No matching passages found. This is NOT evidence of absence, "
                    "read the paper before recording a No.")
        rows.append({"domain": paper.domain, "citation": paper.citation,
                     "filename": paper.filename, "item": item.label,
                     "evidence": evidence, "n_hits": len(snips)})
    return rows


# ---------------------------------------------------------------- verification
def verify_page_one(paper: Paper) -> dict:
    """Protocol step G. Checks the PDF's first page against the intended
    citation before the paper goes anywhere near a score.

    Two of the twelve original manifest entries held the wrong paper and both
    were obvious on page one. The flags below are a SCREEN, not a verdict. In
    one of those two failures the expected surname did appear on page one,
    because the intended author was a coauthor on the wrong paper. Read the
    excerpt and record your own human_verdict."""
    if not paper.path.exists():
        return {"filename": paper.filename, "expected_citation": paper.citation,
                "auto_flag": "MISSING FILE", "author_found": "", "year_found": "",
                "page1_excerpt": f"FILE NOT FOUND at {paper.path}", "human_verdict": ""}

    with pdfplumber.open(paper.path) as pdf:
        first = _norm(pdf.pages[0].extract_text())

    squashed = re.sub(r"\s+", "", first).lower()      # survives space-stripped PDFs
    surname = re.split(r"[ ,]", paper.citation.strip())[0].strip().lower()
    surname = re.sub(r"[^a-z]", "", surname)          # drop punctuation, e.g. "EAGLE/CADDIE"
    author_found = len(surname) > 2 and surname in squashed

    m = re.search(r"\b(?:19|20)\d{2}\b", paper.citation)
    exp_year = m.group(0) if m else ""
    year_found = bool(exp_year) and exp_year in first

    if author_found and year_found:
        flag = "AUTO PASS (confirm by eye)"
    elif author_found or year_found:
        flag = "REVIEW - partial match"
    else:
        flag = "AUTO FAIL - neither author nor year on page 1"

    return {"filename": paper.filename, "expected_citation": paper.citation,
            "auto_flag": flag,
            "author_found": f"{surname or '(none)'}={author_found}",
            "year_found": f"{exp_year or '(none)'}={year_found}",
            "page1_excerpt": first[:VERIFY_EXCERPT_CHARS],
            "human_verdict": ""}

print("Defined _norm, extract_pages (3 variants/page), find_snippets, evaluate_paper, verify_page_one.")

In [ ]:
# Step 1 of 2: verify identity BEFORE gathering evidence.
# Protocol step G. Anything that is not a clean pass should be resolved before
# its evidence is read, let alone scored.
verification = []
for paper in papers:
    v = verify_page_one(paper)
    verification.append(v)
    mark = "ok " if v["auto_flag"].startswith("AUTO PASS") else "!! "
    print(f"  {mark}{v['auto_flag']:<42} {paper.filename}")

flagged = [v for v in verification if not v["auto_flag"].startswith("AUTO PASS")]
print(f"\nVerification: {len(verification) - len(flagged)} auto-pass, {len(flagged)} needing review.")
print("An auto-pass is a screen, not a verdict. Read page1_excerpt in verification_log.csv")
print("and record your own judgement in the human_verdict column.\n")

# Step 2 of 2: gather evidence.
all_results = []
for paper in papers:
    rows = evaluate_paper(paper)
    hits = sum(r["n_hits"] for r in rows)
    empty = sum(1 for r in rows if r["n_hits"] == 0)
    all_results.extend(rows)
    print(f"  {paper.filename:<38} {hits:>3} snippets, {empty:>2}/13 items with no match")

print(f"\nDone. {len(all_results)} rows "
      f"({len(papers)} papers x {len(CHECKLIST_ITEMS)} items = {len(papers) * len(CHECKLIST_ITEMS)}).")

## Write the outputs

Three files, split by what each is for.

`auto_review.xlsx` is the reading aid, with long wrapped snippets meant to be read beside the PDF. It contains no formulas, so it is immune to the cached-value problem that affects formula cells written by openpyxl.

`auto_review.csv` carries identical content in plain text, for grep, diff and version control.

`verification_log.csv` is the page one audit record, sized to be pasted straight into the Screening Log of the tracker.

All three are separate from `checklist_scoring_tracker.xlsx`. Re-running this notebook never touches your scoring.

In [ ]:
FONT_NAME   = "Arial"
HEADER_FILL = PatternFill(start_color="1F3864", end_color="1F3864", fill_type="solid")
HEADER_FONT = Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10)
BODY_FONT   = Font(name=FONT_NAME, size=10)
THIN        = Side(style="thin", color="B7B7B7")
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
WRAP        = Alignment(wrap_text=True, vertical="top", horizontal="left")

HEADERS = ["Domain", "Citation", "Filename", "Checklist Item",
           "Evidence (page: snippet)", "Hits", "My verdict", "My reasoning"]

def row_values(r):
    return [r["domain"], r["citation"], r["filename"], r["item"],
            r["evidence"], r["n_hits"], "", ""]

# ---- auto_review.xlsx : the reading aid. No formulas, so it cannot suffer the
# ---- cached-value problem that affects formula cells written by openpyxl.
wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Auto Review"

for col, text in enumerate(HEADERS, start=1):
    c = ws.cell(row=1, column=col, value=text)
    c.font, c.fill, c.border = HEADER_FONT, HEADER_FILL, BORDER

for i, r in enumerate(all_results, start=2):
    for col, val in enumerate(row_values(r), start=1):
        c = ws.cell(row=i, column=col, value=val)
        c.font, c.alignment, c.border = BODY_FONT, WRAP, BORDER

for col, w in enumerate([12, 40, 30, 22, 78, 6, 12, 40], start=1):
    ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = w
ws.freeze_panes = "A2"
ws.row_dimensions[1].height = 20

OUT_DIR.mkdir(parents=True, exist_ok=True)
wb.save(XLSX_PATH)

# ---- auto_review.csv : same content, greppable and diffable
with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(HEADERS)
    for r in all_results:
        w.writerow(row_values(r))

# ---- verification_log.csv : the page-one audit record
VHEAD = ["filename", "expected_citation", "auto_flag", "author_found",
         "year_found", "page1_excerpt", "human_verdict"]
with open(VERIFY_PATH, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=VHEAD)
    w.writeheader()
    w.writerows(verification)

for pth in (XLSX_PATH, CSV_PATH, VERIFY_PATH):
    print(f"  wrote {pth.name:<24} {pth.stat().st_size:>8,} bytes")
print(f"\n{len(all_results)} evidence rows, {len(verification)} verification rows.")
print("Neither file suggests a verdict. Fill in 'My verdict' yourself, then transfer")
print("confirmed answers into the Scoring tab of checklist_scoring_tracker.xlsx.")

## What this tooling actually contributed

Written for the methods section, so the pipeline is described honestly rather than credited with analytical work it did not do.

**It accelerated reading.** For each paper it located candidate passages across the full PDF and reported them with page numbers, which is faster than reading fifteen papers cover to cover looking for thirteen things in each.

**It assigned no verdicts.** All 195 entries in the Scoring tab were decided by a human reading the full paper. The evidence snippets were a starting point for that reading, never a substitute, and several verdicts rest on passages the patterns never surfaced.

**Two of its thirteen columns originally suggested verdicts, and both were withdrawn as unreliable.** The Intention-to-Treat suggestion was wrong in every case checked. The registration suggestion was wrong wherever the answer was N/A or where the trial had registered outside ClinicalTrials.gov. Those failures are documented in the header cell above, and they are the reason the notebook now claims nothing.

**Its most consequential defect was mechanical, not conceptual.** The original matched against raw extracted text, so a line break inside a phrase or a PDF that extracted without spaces defeated it. That is now fixed by whitespace normalisation, and the fix would have caught every one of the false negatives.

---

## Operating notes

- **Run the verification step first and act on it.** Two of the twelve original manifest entries held the wrong paper, both by the same mechanism, a companion or related study by an overlapping author group. `AUTO PASS` means the expected surname and year both appear on page one. It does not mean the paper is right, and in one of the two failures the expected surname *did* appear because the intended author was a coauthor on the wrong paper. Read `page1_excerpt` and record your own `human_verdict`.
- **"No matching passages found" is not evidence of absence.** Never record a No on that basis alone.
- **Supporting-evidence rows are skipped by design.** The manifest loader ignores rows with a blank Domain, which is how the supporting block at the bottom of the Papers sheet stays out of the run.
- **Each run overwrites all three outputs.** Nothing accumulates and nothing merges with a prior run, so copy anything you have annotated before re-running.
- **Patterns are meant to be tuned.** If a paper phrases something the lists miss, add it. Widening a pattern is cheap; a missed passage is not.